# 클라이언트 측 도구로 고객 상담 에이전트 만들기

이 레시피에서는 Claude 3와 클라이언트 측 도구로 고객 상담 챗봇을 만드는 방법을 보여 줍니다. 이 챗봇은 고객 정보를 조회하고, 주문 상세를 가져오고, 고객을 대신해 주문을 취소할 수 있습니다. 필요한 도구를 정의하고, 챗봇의 동작을 보여 주기 위해 가상의 응답을 시뮬레이션합니다.

## 1단계: 환경 설정

먼저 필요한 라이브러리를 설치하고 Claude API 클라이언트를 설정합니다.

In [ ]:
%pip install anthropic

In [18]:
import anthropic

client = anthropic.Client()
MODEL_NAME = "claude-opus-4-1"

## 2단계: 클라이언트 측 도구 정의하기

다음으로 챗봇이 고객을 응대할 때 사용할 클라이언트 측 도구를 정의합니다. get_customer_info, get_order_details, cancel_order 세 가지 도구를 만듭니다.

In [2]:
tools = [
    {
        "name": "get_customer_info",
        "description": "Retrieves customer information based on their customer ID. Returns the customer's name, email, and phone number.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {
                    "type": "string",
                    "description": "The unique identifier for the customer.",
                }
            },
            "required": ["customer_id"],
        },
    },
    {
        "name": "get_order_details",
        "description": "Retrieves the details of a specific order based on the order ID. Returns the order ID, product name, quantity, price, and order status.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The unique identifier for the order.",
                }
            },
            "required": ["order_id"],
        },
    },
    {
        "name": "cancel_order",
        "description": "Cancels an order based on the provided order ID. Returns a confirmation message if the cancellation is successful.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The unique identifier for the order to be cancelled.",
                }
            },
            "required": ["order_id"],
        },
    },
]

## 3단계: 가상의 도구 응답 시뮬레이션하기

실제 고객 데이터나 주문 정보가 없으므로, 도구의 응답을 가상으로 만들어 시뮬레이션합니다. 실제 상황이라면 이 함수들이 여러분의 실제 고객 데이터베이스 및 주문 관리 시스템과 연동될 것입니다.

In [15]:
def get_customer_info(customer_id):
    # Simulated customer data
    customers = {
        "C1": {"name": "John Doe", "email": "john@example.com", "phone": "123-456-7890"},
        "C2": {"name": "Jane Smith", "email": "jane@example.com", "phone": "987-654-3210"},
    }
    return customers.get(customer_id, "Customer not found")


def get_order_details(order_id):
    # Simulated order data
    orders = {
        "O1": {
            "id": "O1",
            "product": "Widget A",
            "quantity": 2,
            "price": 19.99,
            "status": "Shipped",
        },
        "O2": {
            "id": "O2",
            "product": "Gadget B",
            "quantity": 1,
            "price": 49.99,
            "status": "Processing",
        },
    }
    return orders.get(order_id, "Order not found")


def cancel_order(order_id):
    # Simulated order cancellation
    if order_id in ["O1", "O2"]:
        return True
    else:
        return False

## 4단계: 도구 호출 처리하고 결과 반환하기

Claude가 만든 도구 호출을 처리하고 알맞은 결과를 반환하는 함수를 만듭니다.

In [4]:
def process_tool_call(tool_name, tool_input):
    if tool_name == "get_customer_info":
        return get_customer_info(tool_input["customer_id"])
    elif tool_name == "get_order_details":
        return get_order_details(tool_input["order_id"])
    elif tool_name == "cancel_order":
        return cancel_order(tool_input["order_id"])

## 5단계: 챗봇과 상호작용하기

이제 챗봇과 상호작용하는 함수를 만들겠습니다. 사용자 메시지를 보내고, Claude가 만든 도구 호출을 처리한 뒤, 최종 응답을 사용자에게 돌려줍니다.

In [13]:
import json


def chatbot_interaction(user_message):
    print(f"\n{'=' * 50}\nUser Message: {user_message}\n{'=' * 50}")

    messages = [{"role": "user", "content": user_message}]

    response = client.messages.create(
        model=MODEL_NAME, max_tokens=4096, tools=tools, messages=messages
    )

    print("\nInitial Response:")
    print(f"Stop Reason: {response.stop_reason}")
    print(f"Content: {response.content}")

    while response.stop_reason == "tool_use":
        tool_use = next(block for block in response.content if block.type == "tool_use")
        tool_name = tool_use.name
        tool_input = tool_use.input

        print(f"\nTool Used: {tool_name}")
        print("Tool Input:")
        print(json.dumps(tool_input, indent=2))

        tool_result = process_tool_call(tool_name, tool_input)

        print("\nTool Result:")
        print(json.dumps(tool_result, indent=2))

        messages = [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": response.content},
            {
                "role": "user",
                "content": [
                    {
                        "type": "tool_result",
                        "tool_use_id": tool_use.id,
                        "content": str(tool_result),
                    }
                ],
            },
        ]

        response = client.messages.create(
            model=MODEL_NAME, max_tokens=4096, tools=tools, messages=messages
        )

        print("\nResponse:")
        print(f"Stop Reason: {response.stop_reason}")
        print(f"Content: {response.content}")

    final_response = next(
        (block.text for block in response.content if hasattr(block, "text")),
        None,
    )

    print(f"\nFinal Response: {final_response}")

    return final_response

## 6단계: 챗봇 테스트하기
예시 질의를 몇 개 던져 고객 상담 챗봇을 테스트해 보겠습니다.

In [17]:
chatbot_interaction("Can you tell me the email address for customer C1?")
chatbot_interaction("What is the status of order O2?")
chatbot_interaction("Please cancel order O1 for me.")


User Message: Can you tell me the email address for customer C1?

Initial Response:
Stop Reason: tool_use
Content: [ContentBlock(text='<thinking>The get_customer_info function retrieves a customer\'s name, email, and phone number given their customer ID. To call this function, I need the customer_id parameter. The user provided the customer ID "C1" in their request, so I have the necessary information to make the API call.</thinking>', type='text'), ContentBlockToolUse(id='toolu_019F9JHokMkJ1dHw5BEh28sA', input={'customer_id': 'C1'}, name='get_customer_info', type='tool_use')]

Tool Used: get_customer_info
Tool Input:
{
  "customer_id": "C1"
}

Tool Result:
{
  "name": "John Doe",
  "email": "john@example.com",
  "phone": "123-456-7890"
}

Response:
Stop Reason: end_turn
Content: [ContentBlock(text='The email address for customer C1 (John Doe) is john@example.com.', type='text')]

Final Response: The email address for customer C1 (John Doe) is john@example.com.

User Message: What is 

'Based on the confirmation received, your order O1 has been successfully cancelled. Please let me know if there is anything else I can assist you with.'

이것으로 끝입니다! Claude 3 모델과 클라이언트 측 도구로 고객 상담 챗봇을 만들었습니다. 이 챗봇은 사용자의 요청에 따라 고객 정보를 조회하고, 주문 상세를 가져오고, 주문을 취소할 수 있습니다. 도구 설명과 스키마를 명확히 정의한 덕분에 Claude가 사용 가능한 도구를 제대로 이해하고 활용해 고객을 응대할 수 있습니다.

여기서 더 나아가 실제 고객 데이터베이스 및 주문 관리 시스템과 연동하고, 더 다양한 고객 상담 업무를 처리할 도구를 추가해 이 예제를 확장해 보세요.